# Unidad 44. Facetado y exploración multivariable

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> El facetado repite una visualización para subconjuntos de datos. Ayuda a comparar grupos si se mantienen preguntas, escalas y codificaciones coherentes.


## Resultados de aprendizaje

Al terminar podrás crear `FacetGrid`, `relplot` y `catplot` con paneles; usar `pairplot` de manera selectiva; y explicar por qué la escala compartida, el número de variables y el tamaño de figura importan.

**Prerequisitos:** unidades 39–43 de Seaborn.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

def ruta_datos(nombre):
    for ruta in [Path("datos") / "seaborn" / nombre, Path("..") / "datos" / "seaborn" / nombre]:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")

observaciones = pd.read_csv(ruta_datos("44_observaciones_modelo.csv"))
observaciones.head()


## 1. Una pregunta por conjunto de paneles

Podemos preguntar: **«¿Cambia la relación entre exactitud y latencia según el dispositivo?»**. Dividimos la misma dispersión en columnas. Cada panel mantiene los mismos ejes, de modo que su posición conserva el mismo significado.

Si cada panel tuviera una escala distinta, una distancia visual parecida podría representar magnitudes muy diferentes. Comprueba siempre las escalas antes de comparar.


In [ ]:
relacion_por_dispositivo = sns.relplot(data=observaciones, x="latencia_ms", y="exactitud_pct", hue="configuracion", style="configuracion", col="dispositivo", kind="scatter", height=4, aspect=1)
relacion_por_dispositivo.set_axis_labels("Latencia (ms)", "Exactitud (%)")
relacion_por_dispositivo.figure.suptitle("Relación simulada por dispositivo", y=1.06)
plt.show()


## 2. `FacetGrid`: paneles y función de dibujo

`FacetGrid` separa los datos en una cuadrícula y aplica una función a cada subconjunto con `map_dataframe`. Da más control que `relplot`, pero requiere comprobar que cada panel recibe las mismas variables y que la leyenda sigue siendo legible.

No añadas filas, columnas y colores a la vez sin una pregunta concreta: cada codificación debe aportar una comparación útil.


In [ ]:
cuadricula = sns.FacetGrid(observaciones, col="dispositivo", hue="configuracion", height=4, aspect=1)
cuadricula.map_dataframe(sns.scatterplot, x="latencia_ms", y="exactitud_pct", s=70)
cuadricula.add_legend(title="Configuración")
cuadricula.set_axis_labels("Latencia (ms)", "Exactitud (%)")
cuadricula.figure.suptitle("FacetGrid con la misma dispersión", y=1.06)
plt.show()


## 3. Paneles categóricos y `pairplot`

`catplot` permite comparar una distribución categórica por panel. En el siguiente ejemplo, cada caja resume las observaciones de un dispositivo; los puntos individuales ayudan a no confundir el resumen con todos los valores.

`pairplot` genera una matriz de relaciones entre varias columnas numéricas. Puede revelar patrones iniciales, pero crece muy rápido: tres variables ya producen nueve paneles. Selecciona pocas variables con una pregunta clara, y no confundas una diagonal de distribuciones con una evaluación de un modelo.


In [ ]:
cajas = sns.catplot(data=observaciones, x="configuracion", y="energia_wh", col="dispositivo", kind="box", height=4, aspect=1)
cajas.set_axis_labels("Configuración", "Energía (Wh)")
cajas.figure.suptitle("Energía simulada por configuración y dispositivo", y=1.06)
plt.show()

matriz_pares = sns.pairplot(observaciones, vars=["exactitud_pct", "latencia_ms", "energia_wh"], hue="configuracion", corner=True, diag_kind="hist")
matriz_pares.figure.suptitle("Relaciones exploratorias seleccionadas", y=1.02)
plt.show()


## 4. `PairGrid`: control avanzado y limitado

`PairGrid` permite elegir una función distinta para la diagonal y para las relaciones entre variables. Es útil cuando un `pairplot` no ofrece suficiente control, pero no es un objetivo en sí mismo. En esta unidad basta con reconocer su estructura y no necesitas memorizarla para cada análisis.

## Práctica guiada

Crea un `relplot` de latencia frente a memoria, con paneles por dispositivo y color y estilo por configuración. Añade etiquetas y explica en un comentario por qué no sería correcto atribuir una diferencia visual al dispositivo sin más datos.


In [ ]:
# Completa la práctica guiada aquí.
latencia_memoria = sns.relplot(
    data=observaciones,
    x="memoria_mb",
    y="latencia_ms",
    hue="configuracion",
    style="configuracion",
    col="dispositivo",
    kind="scatter",
    s=75,
    height=4,
    aspect=1.1,
)
latencia_memoria.set_axis_labels("Memoria (MB)", "Latencia (ms)")
latencia_memoria.set_titles("Dispositivo: {col_name}")
latencia_memoria.legend.set_title("Configuración")
latencia_memoria.figure.suptitle(
    "Latencia frente a memoria por dispositivo y configuración", y=1.05
)
plt.show()

# Explicación: No sería correcto atribuir una diferencia visual al dispositivo sin más datos porque
# estas observaciones son simuladas y cada dispositivo podría estar ejecutando cargas de trabajo,
# frecuencias de procesador o restricciones térmicas distintas que no están registradas en la tabla.

- **[U44.1]** `latencia_memoria = sns.relplot(data=observaciones, x="memoria_mb", y="latencia_ms", hue="configuracion", style="configuracion", col="dispositivo", kind="scatter", s=75, height=4, aspect=1.1)` crea una cuadrícula de **nivel de figura** (`FacetGrid`) con un panel por dispositivo (`col="dispositivo"`), escalas X e Y compartidas y doble codificación (`hue` y `style`) para la configuración,
- **[U44.1]** `latencia_memoria.set_axis_labels("Memoria (MB)", "Latencia (ms)")`, `latencia_memoria.set_titles("Dispositivo: {col_name}")` y `latencia_memoria.legend.set_title("Configuración")` configuran las etiquetas exteriores con unidades, los encabezados de cada panel y el título de la leyenda[cite: 7],
- **[U44.1]** `latencia_memoria.figure.suptitle(..., y=1.05)` y `plt.show()` añaden el título superior global sin solapar los paneles y muestran la figura[cite: 7].

## Ejercicios

### Ejercicio 44.01 · Paneles comparables

Comprueba que hay 12 observaciones por dispositivo. Crea un `FacetGrid` de exactitud frente a energía, con color por configuración y las mismas etiquetas en ambos paneles.

### Ejercicio 44.02 · `relplot` con decisión explícita

Crea un `relplot(kind="scatter")` de memoria frente a latencia, con columnas por dispositivo. Guarda el resultado en `memoria_latencia` y explica por qué las escalas compartidas facilitan esta comparación.

### Ejercicio 44.03 · Categorías en paneles

Crea un `catplot(kind="strip")` de exactitud por configuración, con paneles por dispositivo. Indica qué información aporta ver cada punto.

### Ejercicio 44.04 · Pairplot selectivo

Crea un `pairplot` con exactitud, latencia y memoria, con `hue="dispositivo"` y `corner=True`. Guarda el resultado en `pares_dispositivo`. Explica por qué no incluyes todas las columnas numéricas disponibles.

### Ejercicio 44.05 · Miniproyecto: comparación multivariable

Construye dos figuras complementarias: una con paneles por dispositivo y otra que muestre relaciones entre tres variables seleccionadas. Redacta tres conclusiones: una pauta observada, un posible factor alternativo y una limitación de estas observaciones sintéticas.


In [ ]:
# Escribe aquí tu solución de los ejercicios 44.01 a 44.05.
# ==========================================
# Ejercicio 44.01 · Paneles comparables
# ==========================================
conteo_dispositivo = observaciones.groupby("dispositivo").size()
assert (conteo_dispositivo == 12).all(), (
    f"Se esperaban 12 observaciones por dispositivo, pero se obtuvo:\n{conteo_dispositivo}"
)

cuadricula_44 = sns.FacetGrid(
    observaciones,
    col="dispositivo",
    hue="configuracion",
    height=4,
    aspect=1.1,
)
cuadricula_44.map_dataframe(sns.scatterplot, x="energia_wh", y="exactitud_pct", s=75)
cuadricula_44.add_legend(title="Configuración")
cuadricula_44.set_axis_labels("Energía (Wh)", "Exactitud (%)")
cuadricula_44.set_titles("Dispositivo: {col_name}")
cuadricula_44.figure.suptitle("44.01 · Exactitud frente a energía por dispositivo", y=1.05)
plt.show()

# ==========================================
# Ejercicio 44.02 · relplot con decisión explícita
# ==========================================
memoria_latencia = sns.relplot(
    data=observaciones,
    kind="scatter",
    x="latencia_ms",
    y="memoria_mb",
    col="dispositivo",
    hue="configuracion",
    style="configuracion",
    s=75,
    height=4,
    aspect=1.1,
)
memoria_latencia.set_axis_labels("Latencia (ms)", "Memoria (MB)")
memoria_latencia.set_titles("Dispositivo: {col_name}")
memoria_latencia.legend.set_title("Configuración")
memoria_latencia.figure.suptitle("44.02 · Memoria frente a latencia por dispositivo", y=1.05)
plt.show()
# Explicación: Mantener escalas compartidas en los ejes X e Y garantiza que una misma distancia
# visual horizontal o vertical represente exactamente la misma magnitud en ambos paneles,
# permitiendo comparar posiciones y pendientes directamente sin distorsiones visuales.

# ==========================================
# Ejercicio 44.03 · Categorías en paneles
# ==========================================
puntos_categoria = sns.catplot(
    data=observaciones,
    x="configuracion",
    y="exactitud_pct",
    col="dispositivo",
    kind="strip",
    jitter=0.12,
    size=7,
    height=4,
    aspect=1.05,
)
puntos_categoria.set_axis_labels("Configuración", "Exactitud (%)")
puntos_categoria.set_titles("Dispositivo: {col_name}")
puntos_categoria.figure.suptitle(
    "44.03 · Observaciones individuales de exactitud por configuración y dispositivo", y=1.05
)
plt.show()
# Qué aporta ver cada punto: Permite inspeccionar el tamaño real de cada subgrupo (6 puntos),
# detectar agrupaciones o valores extremos y evitar que un resumen estadístico oculte la
# dispersión real dentro de cada combinación.

# ==========================================
# Ejercicio 44.04 · Pairplot selectivo
# ==========================================
pares_dispositivo = sns.pairplot(
    observaciones,
    vars=["exactitud_pct", "latencia_ms", "memoria_mb"],
    hue="dispositivo",
    corner=True,
    diag_kind="hist",
)
pares_dispositivo.figure.suptitle(
    "44.04 · Matriz de relaciones seleccionadas por dispositivo", y=1.02
)
plt.show()
# Explicación: No se incluyen todas las columnas numéricas disponibles porque el número de paneles
# crece cuadráticamente con el número de variables, dificultando la lectura y mezclando métricas
# sin una pregunta exploratoria concreta.

# ==========================================
# Ejercicio 44.05 · Miniproyecto: comparación multivariable
# ==========================================
# Figura 1: Paneles por dispositivo (relación entre energía, exactitud y configuración)
fig_panel = sns.relplot(
    data=observaciones,
    x="energia_wh",
    y="exactitud_pct",
    hue="configuracion",
    style="configuracion",
    col="dispositivo",
    kind="scatter",
    s=80,
    height=4,
    aspect=1.1,
)
fig_panel.set_axis_labels("Energía (Wh)", "Exactitud (%)")
fig_panel.set_titles("Dispositivo: {col_name}")
fig_panel.legend.set_title("Configuración")
fig_panel.figure.suptitle(
    "44.05 (1/2) · Exactitud y consumo energético por dispositivo y configuración", y=1.05
)
plt.show()

# Figura 2: Tres variables seleccionadas (exactitud, latencia y energía) separadas por configuración
fig_tres_vars = sns.pairplot(
    observaciones,
    vars=["exactitud_pct", "latencia_ms", "energia_wh"],
    hue="configuracion",
    corner=True,
    diag_kind="hist",
)
fig_tres_vars.figure.suptitle(
    "44.05 (2/2) · Relaciones entre exactitud, latencia y energía por configuración", y=1.02
)
plt.show()

### Explicación línea por línea (Ejercicios 44.01 a 44.05)

- **[U44.1]** `conteo_dispositivo = observaciones.groupby("dispositivo").size()` y `assert (conteo_dispositivo == 12).all(), ...` verifican que cada categoría de `dispositivo` contiene exactamente 12 observaciones antes de dividir en paneles[cite: 7],
- **[U44.2]** `cuadricula_44 = sns.FacetGrid(observaciones, col="dispositivo", hue="configuracion", height=4, aspect=1.1)` y `cuadricula_44.map_dataframe(sns.scatterplot, x="energia_wh", y="exactitud_pct", s=75)` construyen una cuadrícula personalizada de nivel de figura mapeando un diagrama de dispersión en cada subconjunto[cite: 7],
- **[U44.2]** `cuadricula_44.add_legend(title="Configuración")` y `cuadricula_44.set_axis_labels("Energía (Wh)", "Exactitud (%)")` añaden la leyenda común y las etiquetas con unidades[cite: 7],
- **[U44.1]** `memoria_latencia = sns.relplot(..., kind="scatter", x="latencia_ms", y="memoria_mb", col="dispositivo", ...)` genera paneles comparables con escalas X e Y compartidas para que las distancias visuales sean directamente comparables[cite: 7],
- **[U44.3]** `puntos_categoria = sns.catplot(..., kind="strip", jitter=0.12, ...)` muestra cada punto individual por configuración y dispositivo evitando ocultar la dispersión interna[cite: 7],
- **[U44.3]** `pares_dispositivo = sns.pairplot(observaciones, vars=["exactitud_pct", "latencia_ms", "memoria_mb"], hue="dispositivo", corner=True, diag_kind="hist")` limita la exploración multivariable a tres métricas concretas y oculta el triángulo superior duplicado (`corner=True`)[cite: 7].

---

### Conclusiones del Ejercicio 44.05
1. **Pauta observada:** En estas observaciones simuladas, los dispositivos se separan claramente en consumo energético y latencia (el entorno `edge` opera con menor energía y mayor latencia, mientras `servidor` consume más energía pero responde más rápido), y dentro de cada dispositivo la configuración `ajustada` tiende a alcanzar mayor exactitud[cite: 7].
2. **Factor alternativo:** Las diferencias observadas entre configuraciones o dispositivos podrían estar condicionadas por el tamaño de lote, la frecuencia de reloj o la carga concurrente del entorno de prueba no incluidos en estas columnas[cite: 7].
3. **Limitación:** Se trata de una muestra sintética reducida (12 filas por dispositivo), por lo que estas relaciones describen únicamente la tabla actual y no permiten predecir el comportamiento en producción[cite: 7].

## Autoevaluación y cierre

1. ¿Qué debe mantenerse para que dos paneles sean comparables?
2. ¿Cuándo elegirías `FacetGrid` frente a `relplot`?
3. ¿Qué aporta `catplot` con paneles?
4. ¿Por qué limitar las variables de un `pairplot`?
5. ¿Qué puede explorar un `PairGrid` sin demostrar causalidad?

La siguiente unidad reunirá estos recursos en un flujo reproducible de EDA y comunicación.
